In [1]:
from pyspark.sql import SparkSession
from pyspark.ml.fpm import FPGrowth
import pyspark.sql.functions as F

# Initialize local Spark
spark = (
    SparkSession.builder.appName("Pediatric_ADR_Mining_Local")
    .config("spark.driver.memory", "18g")
    .config("spark.executor.memory", "4g")
    .config("spark.driver.maxResultSize", "4g")
    .config("spark.sql.execution.arrow.pyspark.enabled", "true")
    .getOrCreate()
)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/05/25 14:50:17 WARN Utils: Your hostname, localhost.localdomain, resolves to a loopback address: 127.0.0.1; using 192.168.8.160 instead (on interface wlp0s20f3)
26/05/25 14:50:17 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/25 14:50:18 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
fp = 'arm_transactions_2026052514.parquet'
df = spark.read.parquet(f"./{fp}")

In [3]:
df.count()

129122

In [ ]:
# FP-Growth at the "Signal Detection" threshold
fpGrowth = FPGrowth(
    itemsCol="adr_case",
    minSupport=0.0002,
    minConfidence=0.7,  # Lower confidence to catch more associations
)

model = fpGrowth.fit(df)

In [ ]:
# Get the rules
rules = model.associationRules

In [ ]:
# IMPORTANT: Filter for actual insights (Consequents should be Reactions)
# This removes the "noise" of demographic-to-demographic rules
interesting_rules = rules.filter(
    F.array_join("consequent", ",").contains("reaction_")
    | F.array_join("consequent", ",").contains("outc_")
).sort(F.desc("lift"))

In [ ]:
interesting_rules.show(20, truncate=False)

In [ ]:
interesting_rules.count()

In [ ]:
interesting_rules.write.mode("overwrite").format("parquet").save("mined_rules/arm_mined_2026052514.parquet")

In [ ]:
spark.stop()